# Base-Qwen no-LoRA ablation — generate on Colab (no tunnel)

Produces base **Qwen2.5-3B-Instruct** (no LoRA) replies for the Persona-RAG *"the tics had to be learned"* control, on a free Colab **T4 GPU**. It runs the model **offline with `transformers`** (Colab's pre-installed, CUDA-matched torch — no vLLM, no endpoint, no tunnel, no connection to your Mac), writes `base_qwen_pairs.jsonl`, and downloads it. Drop that file into the repo and Claude scores it with the **exact** paper metrics.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. In step 2 you will upload `data/finetune/eval.jsonl` (the held-out split, from the repo).

### 1. Install (transformers — Colab's torch, no CUDA rebuild)

In [ ]:
!pip -q install -U transformers accelerate
# Uses Colab's pre-installed torch/CUDA -> avoids the vLLM 'libcudart.so.13' version mismatch.

### 2. Upload the held-out split

In [ ]:
from google.colab import files
print('Upload data/finetune/eval.jsonl from the repo:')
up = files.upload()
EVAL_PATH = list(up.keys())[0]
print('got:', EVAL_PATH)

### 3. Generate base-Qwen replies (~5–15 min: model download + 300 decodes)

In [ ]:
import json, random, statistics as st, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL = 'Qwen/Qwen2.5-3B-Instruct'
tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = 'left'                       # left-pad for correct batched decoding
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16).to('cuda').eval()

# Parse eval.jsonl like compare_persona._load_sharegpt (system/human/gpt triples).
records = []
for line in open(EVAL_PATH, encoding='utf-8'):
    line = line.strip()
    if not line:
        continue
    by_role = {m['from']: m['value'] for m in json.loads(line)['conversations']}
    if 'human' in by_role and 'gpt' in by_role:
        records.append({'system': by_role.get('system', ''), 'human': by_role['human'], 'gpt': by_role['gpt']})

# Match `make compare`: seed 0, n 300, same shuffle as compare_persona._sample.
rng = random.Random(0)
rng.shuffle(records)
records = records[:300]
print(len(records), 'items (seed 0) -- identical thin prompt + items as the controlled arm')

def to_prompt(r):
    msgs = [{'role': 'system', 'content': r['system']}, {'role': 'user', 'content': r['human']}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

gen_base, B = [], 8                             # lower B to 4 if you hit CUDA OOM
for start in range(0, len(records), B):
    prompts = [to_prompt(r) for r in records[start:start + B]]
    enc = tok(prompts, return_tensors='pt', padding=True, truncation=True, max_length=2048).to('cuda')
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=200, do_sample=True, temperature=0.8,
                             top_p=1.0, pad_token_id=tok.pad_token_id)
    new = out[:, enc['input_ids'].shape[1]:]
    gen_base.extend(t.strip() for t in tok.batch_decode(new, skip_special_tokens=True))
    print(len(gen_base), '/', len(records), 'done')

with open('base_qwen_pairs.jsonl', 'w', encoding='utf-8') as f:
    for i, (r, g) in enumerate(zip(records, gen_base)):
        print(json.dumps({'item_id': i, 'real': r['gpt'], 'gen_base': g}, ensure_ascii=False), file=f)

print('wrote base_qwen_pairs.jsonl (', len(gen_base), 'rows )')
print('mean base reply length (chars):', round(st.mean(len(g) for g in gen_base), 1))
print('sanity -- 2 base-Qwen outputs (model text only):')
for g in gen_base[:2]:
    print('  -', repr(g[:120]))

### 4. Download the result

In [ ]:
from google.colab import files
files.download('base_qwen_pairs.jsonl')

## Hand it back to Claude

1. The download `base_qwen_pairs.jsonl` (~300 lines: your real reply + the base-model reply per item) stays on your machine.
2. Put it in the repo at: `data/eval/compare/base_qwen/pairs.jsonl`
3. Tell Claude **"base-qwen pairs are in"** — it runs
   `uv run python scripts/base_qwen_arm.py --from-pairs data/eval/compare/base_qwen/pairs.jsonl`
   to score it with the exact paper metrics and add the **base** column to the Arm-B table.

*(Prefer not to move the file? Paste the printed sanity lines for a first look — but the file gives the exact, paper-consistent numbers.)*